# 第 6 章｜异步子 Agent 的完整生命周期

本实验从 Notebook 启动一个本地 LangGraph Agent Server。`supervisor` 通过 `AsyncSubAgent` 的 **ASGI** 路径委派给故意等待 8 秒的 `researcher`。我们读取真实工具调用、任务 ID、服务端 run 状态，并验证启动、查询、列举、更新和取消。

**运行方式**：在仓库根目录执行，依赖与命令见 [`notebooks/README.md`](../README.md)。默认读取仓库根目录未提交的 `.env` 或进程环境变量。设置 `MODEL_API_KEY`（也接受 `DEEPSEEK_API_KEY` 或第 1 章的 `SILICONFLOW_API_KEY`）会调用真实模型；未设置时使用脚本模型，只控制 supervisor 选择哪个工具，**不会**模拟工具、任务或服务端。脚本模式不能证明真实模型会遵从指令。随附的运行输出来自脚本模式；端口和任务 ID 每次运行都会变化。

## 运行环境

本 Notebook 在 macOS（Apple Silicon）、Python 3.12.13 下验证。依赖版本为 `deepagents==0.7.15`、`langchain==1.4.2`、`langgraph==1.2.11`、`langchain-openai==1.6.2`、`langgraph-cli==0.4.32`（含 `[inmem]`）和 `langgraph-sdk==0.4.5`。安装与内核选择见 [Notebook 索引](../README.md)。本章会自行启动服务，无需先运行第 1 章或手动预启动服务。

In [1]:
import asyncio
import importlib.metadata as metadata
import json
import os
import platform
import shutil
import socket
import subprocess
import sys
import tempfile
import time
from pathlib import Path

from dotenv import load_dotenv
from langgraph_sdk import get_client

ROOT = next((path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (path / "content/ch06-async-subagents.md").exists()), None)
assert ROOT is not None, "请从仓库目录运行 Notebook"
DEMO = ROOT / "notebooks/ch06/demo"
load_dotenv(ROOT / ".env", override=False)
MODEL_KEY = os.getenv("MODEL_API_KEY") or os.getenv("DEEPSEEK_API_KEY") or os.getenv("SILICONFLOW_API_KEY")
SILICONFLOW = not (os.getenv("MODEL_API_KEY") or os.getenv("DEEPSEEK_API_KEY")) and bool(os.getenv("SILICONFLOW_API_KEY"))
SCRIPTED = not bool(MODEL_KEY)
print("supervisor 模式：", "脚本模型（无模型 API）" if SCRIPTED else "真实模型 API")
print("操作系统：", platform.system(), platform.machine())
print("Python：", sys.version.split()[0])
for package in ("deepagents", "langchain", "langgraph", "langchain-openai", "langgraph-cli", "langgraph-sdk"):
    print(f"{package}=={metadata.version(package)}")

supervisor 模式： 脚本模型（无模型 API）
操作系统： Darwin arm64
Python： 3.12.13
deepagents==0.7.15
langchain==1.4.2
langgraph==1.2.11
langchain-openai==1.6.2
langgraph-cli==0.4.32
langgraph-sdk==0.4.5


## 启动与调用辅助函数

服务端只绑定 `127.0.0.1`。就绪检查同时观察子进程是否提前退出；所有等待均有截止时间。实验的 `finally` 会取消仍在运行的子任务、删除本次创建的 thread，并关闭服务端。

In [2]:
def unused_local_port():
    with socket.socket() as sock:
        sock.bind(("127.0.0.1", 0))
        return sock.getsockname()[1]


async def wait_ready(client, process, log_path, timeout=30):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        if process.poll() is not None:
            raise RuntimeError(f"Agent Server 提前退出（exit={process.returncode}），日志：{log_path}")
        try:
            assistants = await asyncio.wait_for(client.assistants.search(), timeout=2)
            if {"supervisor", "researcher"} <= {item["graph_id"] for item in assistants}:
                return
        except Exception:
            pass  # 启动期间连接失败是预期状态；到截止时间后报错。
        await asyncio.sleep(0.3)
    raise TimeoutError(f"Agent Server 未在 {timeout}s 内就绪，日志：{log_path}")


async def ask(client, thread_id, text, expected_tool, show=True):
    state = await asyncio.wait_for(
        client.runs.wait(
            thread_id, "supervisor", input={"messages": [{"role": "user", "content": text}]}
        ),
        timeout=30,
    )
    human_indices = [i for i, msg in enumerate(state["messages"]) if msg["type"] == "human"]
    recent = state["messages"][human_indices[-1]:]
    calls = [call["name"] for msg in recent for call in msg.get("tool_calls", [])]
    assert calls == [expected_tool], f"预期只调用 {expected_tool}，实际：{calls}"
    outputs = [msg["content"] for msg in recent if msg["type"] == "tool"]
    assert len(outputs) == 1, f"预期一条工具结果，实际：{outputs}"
    if show:
        print(expected_tool, "→", json.loads(outputs[0]) if expected_tool == "check_async_task" else outputs[0])
    return state, outputs[0]


async def wait_task(client, parent_id, task_id, timeout=40):
    deadline = time.monotonic() + timeout
    last_status = None
    while time.monotonic() < deadline:
        state, output = await ask(client, parent_id, f"CHECK|{task_id}", "check_async_task", show=False)
        status = state["async_tasks"][task_id]["status"]
        if status != last_status:
            print("check_async_task →", json.loads(output))
            last_status = status
        if status == "success":
            return state, output
        if status in {"error", "cancelled", "interrupted", "timeout"}:
            raise RuntimeError(f"子任务异常结束：{status}; {output}")
        await asyncio.sleep(1)
    raise TimeoutError(f"任务 {task_id} 在 {timeout}s 内未完成")

## 从启动到清理，一次执行

三个独立的后台任务分别用于：完成并读取结果、在原 task ID 上追加指令、取消运行中的任务。断言使用工具调用和 state/run 数据，不依赖模型回答的措辞。

In [3]:
async def experiment():
    cli = Path(sys.executable).with_name("langgraph")
    assert cli.is_file(), f"未找到 {cli}；请安装 langgraph-cli[inmem]"
    port = unused_local_port()
    log = tempfile.NamedTemporaryFile(mode="w", prefix="ch06-agent-server-", suffix=".log", delete=False)
    log_path = Path(log.name)
    server_dir = tempfile.TemporaryDirectory(prefix="ch06-demo-")
    env = os.environ.copy()
    if SCRIPTED:
        env["CH06_SCRIPTED_MODEL"] = "1"
    else:
        env.pop("CH06_SCRIPTED_MODEL", None)
        env["MODEL_API_KEY"] = MODEL_KEY
        if SILICONFLOW:
            env.setdefault("MODEL_BASE_URL", os.getenv("SILICONFLOW_BASE_URL", "https://api.siliconflow.cn/v1"))
            env.setdefault("MODEL_NAME", "Qwen/Qwen2.5-7B-Instruct")
    process = None
    client = get_client(url=f"http://127.0.0.1:{port}")
    parent_id = None
    task_ids = set()
    cleanup_errors = []
    successful = False
    try:
        shutil.copytree(DEMO, server_dir.name, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns(".langgraph_api", "__pycache__"))
        process = subprocess.Popen(
            [str(cli), "dev", "--no-browser", "--no-reload", "--port", str(port),
             "--n-jobs-per-worker", "4", "--config", "langgraph.json"],
            cwd=server_dir.name, env=env, stdout=log, stderr=subprocess.STDOUT,
        )
        await wait_ready(client, process, log_path)
        print("Agent Server 已就绪：", f"http://127.0.0.1:{port}")
        parent_id = (await client.threads.create())["thread_id"]

        # 1. 启动立即产生 task ID；后台 run 尚在执行。
        state, output = await ask(client, parent_id, "START|总结异步子 Agent 的状态变化", "start_async_task")
        first_id = next(iter(state["async_tasks"]))
        task_ids.add(first_id)
        first = state["async_tasks"][first_id]
        assert first_id in output and first["status"] == "running"
        first_run = await client.runs.get(first_id, first["run_id"])
        assert first_run["status"] in {"pending", "running"}, first_run["status"]
        print("主 Agent 已返回，子 run 状态：", first_run["status"])

        # 2. 列举和轮询直到完成；确认服务端结果与 task ID。
        _, listing = await ask(client, parent_id, "LIST", "list_async_tasks")
        assert first_id in listing
        state, result = await wait_task(client, parent_id, first_id)
        assert "Research completed" in json.loads(result)["result"]
        assert (await client.runs.get(first_id, state["async_tasks"][first_id]["run_id"]))["status"] == "success"

        # 3. 更新另一任务：task ID 保持不变，run ID 更新，最终结果含新指令。
        state, _ = await ask(client, parent_id, "START|撰写一份简短摘要", "start_async_task")
        second_id = (set(state["async_tasks"]) - task_ids).pop()
        task_ids.add(second_id)
        old_run = state["async_tasks"][second_id]["run_id"]
        state, output = await ask(
            client, parent_id, f"UPDATE|{second_id}|追加要求：使用三条要点", "update_async_task"
        )
        updated = state["async_tasks"][second_id]
        assert second_id in output and updated["run_id"] != old_run
        state, result = await wait_task(client, parent_id, second_id)
        assert "追加要求：使用三条要点" in json.loads(result)["result"]

        # 4. 取消第三个任务，并观察服务端进入结束态。
        state, _ = await ask(client, parent_id, "START|准备一个将被取消的任务", "start_async_task")
        third_id = (set(state["async_tasks"]) - task_ids).pop()
        task_ids.add(third_id)
        state, output = await ask(client, parent_id, f"CANCEL|{third_id}", "cancel_async_task")
        assert third_id in output and state["async_tasks"][third_id]["status"] == "cancelled"
        deadline = time.monotonic() + 10
        while time.monotonic() < deadline:
            server_status = (await client.runs.get(third_id, state["async_tasks"][third_id]["run_id"]))["status"]
            if server_status in {"interrupted", "cancelled"}:
                break
            await asyncio.sleep(0.25)
        else:
            raise TimeoutError("服务端未确认取消")
        _, listing = await ask(client, parent_id, "LIST", "list_async_tasks")
        assert all(task_id in listing for task_id in task_ids)
        print("三个任务的生命周期验证完成；取消后的服务端状态：", server_status)
        successful = True
        return {"parent_id": parent_id, "task_ids": sorted(task_ids), "cancel_status": server_status}
    finally:
        if process is not None:
            # 即使某个断言失败，也通过服务端查找本次主线程跟踪到的任务。
            if parent_id is not None:
                try:
                    tracked = (await client.threads.get(parent_id)).get("values", {}).get("async_tasks", {})
                    task_ids.update(tracked)
                    for task_id in task_ids:
                        task = tracked.get(task_id)
                        if task:
                            try:
                                run = await client.runs.get(task_id, task["run_id"])
                                if run["status"] in {"pending", "running"}:
                                    await client.runs.cancel(task_id, task["run_id"])
                            except Exception as error:
                                cleanup_errors.append(f"run {task_id}: {type(error).__name__}")
                        try:
                            await client.threads.delete(task_id)
                        except Exception as error:
                            cleanup_errors.append(f"thread {task_id}: {type(error).__name__}")
                    await client.threads.delete(parent_id)
                except Exception as error:
                    cleanup_errors.append(f"主 thread: {type(error).__name__}")
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=5)
        server_dir.cleanup()
        log.close()
        if successful and not cleanup_errors:
            log_path.unlink(missing_ok=True)
        else:
            print("失败诊断日志：", log_path)
        if cleanup_errors:
            raise RuntimeError("清理未完成：" + "; ".join(cleanup_errors) + f"；日志：{log_path}")
        print("本次创建的任务与 Agent Server 已清理")


summary = await experiment()
summary

Agent Server 已就绪： http://127.0.0.1:56427


start_async_task → Launched async subagent. task_id: 01a0d8a1-00a8-7313-9114-4956b4d588ee
主 Agent 已返回，子 run 状态： pending


list_async_tasks → 1 tracked task(s):
- task_id: 01a0d8a1-00a8-7313-9114-4956b4d588ee  agent: researcher  status: running


check_async_task → {'status': 'running', 'thread_id': '01a0d8a1-00a8-7313-9114-4956b4d588ee'}


check_async_task → {'status': 'success', 'thread_id': '01a0d8a1-00a8-7313-9114-4956b4d588ee', 'result': 'Research completed: 总结异步子 Agent 的状态变化'}


...

check_async_task → {'status': 'success', 'thread_id': '01a0d8a1-2ba3-7e72-9f89-545c82845ae0', 'result': 'Research completed: 追加要求：使用三条要点'}


start_async_task → Launched async subagent. task_id: 01a0d8a1-56b6-7b00-8c33-ee5ccab37bad


cancel_async_task → Cancelled async subagent task: 01a0d8a1-56b6-7b00-8c33-ee5ccab37bad


list_async_tasks → 3 tracked task(s):
- task_id: 01a0d8a1-00a8-7313-9114-4956b4d588ee  agent: researcher  status: success
- task_id: 01a0d8a1-2ba3-7e72-9f89-545c82845ae0  agent: researcher  status: success
- task_id: 01a0d8a1-56b6-7b00-8c33-ee5ccab37bad  agent: researcher  status: cancelled
三个任务的生命周期验证完成；取消后的服务端状态： interrupted
本次创建的任务与 Agent Server 已清理


{'parent_id': '01a0d8a0-fdbd-71d1-940f-542c08e4e332',
 'task_ids': ['01a0d8a1-00a8-7313-9114-4956b4d588ee',
  '01a0d8a1-2ba3-7e72-9f89-545c82845ae0',
  '01a0d8a1-56b6-7b00-8c33-ee5ccab37bad'],
 'cancel_status': 'interrupted'}

## 观察结论与边界

- `start_async_task` 返回的 task ID 是子 thread ID；启动时的服务端 run 仍为 `pending` 或 `running`。
- `check_async_task` 读取 run 的最新状态；成功后再从子 thread 取结果。轮询有截止时间，超时会报错并触发清理。
- `update_async_task` 沿用 task ID，创建新的 run；`cancel_async_task` 同时更新主 Agent 的跟踪状态与服务端 run。
- `list_async_tasks` 展示主 Agent 已跟踪的任务。Notebook 只处理本次创建的 thread；每次都在临时服务目录运行，关闭后删除本次的服务状态文件。
- 脚本模式验证服务、ASGI、中间件与 SDK 的实际调用链。配置模型 Key 后请再次从头执行，以验证所选模型对五个工具的选择。

**常见错误**：找不到 `langgraph` 命令时按索引安装 `langgraph-cli[inmem]`；服务提前退出或就绪超时会给出临时日志路径；真实模型认证或工具选择失败会抛出异常，先检查模型 Key、`MODEL_BASE_URL` 和模型是否支持工具调用。对已结束的任务再 `check` 会返回结束状态；取消后的服务端 run 通常显示 `interrupted`，主 Agent 的跟踪状态显示 `cancelled`。